In [ ]:
import os
import sys
from pathlib import Path

# Run from the xLSTM-Mixer project root so outputs/checkpoints resolve correctly.
XLSTM_ROOT = (Path.cwd().parent / "models" / "xlstm-mixer").resolve()
os.chdir(XLSTM_ROOT)
sys.path.insert(0, str(XLSTM_ROOT))


In [ ]:
from pathlib import Path

import wandb
from lightning.pytorch.callbacks import StochasticWeightAveraging

from xlstm_mixer.cli_helper import LoggerSaveConfigCallback, TaskCLI
from xlstm_mixer.exp.exp import ForecastingExp
from xlstm_mixer.lit.data import TSLibDataModule

# User-specified arguments; everything else uses TaskCLI / LightningCLI defaults.
dataset = "Electricity"
pred_len = 96
seq_len = 96
lr = 0.0005
batch_size = 32
init_token = 3
xlstm_embedding_dim = 1024
slstm_conv1d_kernel_size = 4
xlstm_num_blocks = 2
slstm_num_heads = 16
xlstm_dropout = 0.1
gamma = 0.99
cosine_epochs = 5
warmup_epochs = 2
constant_gamma_epochs = 1
seed = 42
max_epochs = 10
fast_dev_run = False
root_path = "../../datasets"

rest_args = [
    "--data", "ForecastingTSLibDataModule",
    "--data.dataset_name", dataset,
    "--data.root_path", root_path,
    "--optimizer.lr", str(lr),
    "--data.seq_len", str(seq_len),
    "--data.pred_len", str(pred_len),
    "--data.label_len", "0",
    "--data.batch_size", str(batch_size),
    "--data.num_workers", "4",
    "--data.persistent_workers", "true",
    "--model", "LongTermForecastingExp",
    "--model.criterion", "torch.nn.L1Loss",
    "--model.architecture", "xLSTMMixer",
    "--model.architecture.num_mem_tokens", str(init_token),
    "--model.architecture.xlstm_num_heads", str(slstm_num_heads),
    "--model.architecture.xlstm_num_blocks", str(xlstm_num_blocks),
    "--model.architecture.xlstm_embedding_dim", str(xlstm_embedding_dim),
    "--model.architecture.xlstm_conv1d_kernel_size", str(slstm_conv1d_kernel_size),
    "--model.architecture.xlstm_dropout", str(xlstm_dropout),
    "--lr_scheduler.constant_gamma_epochs", str(constant_gamma_epochs),
    "--lr_scheduler.gamma", str(gamma),
    "--lr_scheduler.cosine_epochs", str(cosine_epochs),
    "--lr_scheduler.warmup_epochs", str(warmup_epochs),
    "--trainer.logger.name", f"{dataset}_xlstm-mixer_{pred_len}_{seed}",
    "--trainer.logger.project", "xlstm-mixer",
    "--trainer.max_epochs", str(max_epochs),
    "--seed_everything", str(seed),
    "--trainer.fast_dev_run", str(fast_dev_run).lower(),
]

cli = TaskCLI(
    ForecastingExp,
    TSLibDataModule,
    subclass_mode_data=True,
    subclass_mode_model=True,
    run=False,
    args=rest_args,
    save_config_callback=LoggerSaveConfigCallback,
)
cli.datamodule.root_path = Path(root_path)
cli.trainer.fit(cli.model, cli.datamodule)
cli.trainer.callbacks = [
    cb
    for cb in cli.trainer.callbacks
    if not isinstance(cb, StochasticWeightAveraging)
]
if cli.trainer.fast_dev_run:
    print("Fast dev run, skipping test")
    print(cli.trainer.logged_metrics)
else:
    cli.trainer.test(
        cli.model,
        cli.datamodule,
        ckpt_path=cli.trainer.checkpoint_callback.best_model_path,
    )
    wandb.finish()
    if "test/MeanSquaredError" not in cli.trainer.logged_metrics:
        mae = cli.trainer.logged_metrics["test/MeanAbsoluteError"].item()
        mape = cli.trainer.logged_metrics["test/MeanAbsolutePercentageError"].item()
        rmse = cli.trainer.logged_metrics["test/RootMeanSquaredError"].item()
        results = {"mae": mae, "mape": mape, "rmse": rmse}
    else:
        mse_test = cli.trainer.logged_metrics["test/MeanSquaredError"].item()
        mae_test = cli.trainer.logged_metrics["test/MeanAbsoluteError"].item()
        results = {"mse_test": mse_test, "mae_test": mae_test}
    print(results)